In [31]:
import pandas as pd
from pathlib import Path

DF_PATH = Path("../data/processed/telco_customer_churn_cleaned.parquet")
df = pd.read_parquet(DF_PATH)

In [32]:
# look at the df
df.head()

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950001,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.849998,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.299999,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.699997,151.65,Yes


In [33]:
# shape of it
df.shape

(7043, 20)

In [34]:
# check NaN values
df.isna().sum()

Gender              0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [35]:
# info
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   int8    
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   Tenure            7043 non-null   int8    
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 non-null   c

In [36]:
# number of services
services = ["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
positive_values = {
    "Yes",
    "DSL",
    "Fiber optic"
}

df["NumServices"] = df[services].apply(lambda row: sum(val in positive_values for val in row), axis=1)

In [37]:
# long time customer
df["LongTimeCustomer"] = (df.Tenure > df.Tenure.quantile(q=0.90)).astype("int8")

In [38]:
# high spender
df["IsHighSpender"] = (df.TotalCharges > df.TotalCharges.quantile(q=0.90)).astype("int8")

In [39]:
# new customer
df["IsNewCustomer"] = (df.Tenure < 6).astype("int8")

In [40]:
# tenure group
bins = [0, 12, 24, 48, 72]
tenure_labels = ["0-12", "12-24", "24-48", "48-72"]
df["TenureGroup"] = pd.cut(df.Tenure, bins=bins, labels=tenure_labels, include_lowest=True).astype("category")

In [41]:
# auto payment
df["AutoPayment"] = (df.PaymentMethod.str.contains("automatic")).astype("int8")

In [42]:
# has support services: customers in support/security often churn less
df["HasSupportServices"] = ((df.OnlineSecurity == "Yes") | (df.TechSupport == "Yes") | (df.DeviceProtection == "Yes")).astype("int8")

In [43]:
# move target to end
col_to_move = df.pop("Churn")
df.insert(len(df.columns), "Churn", col_to_move)

In [44]:
# see final df
df.head()

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,MonthlyCharges,TotalCharges,NumServices,LongTimeCustomer,IsHighSpender,IsNewCustomer,TenureGroup,AutoPayment,HasSupportServices,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,29.850000,29.85,2,0,0,1,0-12,0,0,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,56.950001,1889.50,4,0,0,0,24-48,0,1,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,53.849998,108.15,4,0,0,1,0-12,0,1,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,42.299999,1840.75,4,0,0,0,24-48,1,1,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,70.699997,151.65,2,0,0,1,0-12,0,0,Yes


In [45]:
# save feature engineered dataframe
df.to_parquet("../data/processed/telco_customer_churn_feature_engineered.parquet", index=False)